[![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://github.com/codespaces)


<!-- gen:values.yml#weeks.05.notebook-title|Demonstration -->
# Week 5 Demonstration: Advanced pandas
<!-- /gen -->

*Examples of reshaping, `MultiIndex`, window functions, and complex aggregation, this time on client data. Run every cell and try every [AI Lens](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-ai-lens) prompt with your [Associate](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-associate).*

## The client

**Hartwell Garden Centres** run three sites (Egham, Windsor, and Staines), each selling three product lines: **Plants**, **Tools**, and the **Café**. We have a year of daily sales and the monthly report their finance office sends by email.

Run the data-setup cell below: It generates the files this notebook reads, so the notebook runs the same wherever you open it, with nothing to download.

In [ ]:
# --- Data setup: Run this cell first (runs anywhere, no download needed) ---
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = "assets/data/week05"
os.makedirs(DATA_DIR, exist_ok=True)
rng = np.random.default_rng(5813 + 1)  # derived from the module seed

CENTRES = ["Egham", "Windsor", "Staines"]
LINES = ["Plants", "Tools", "Café"]
dates = pd.date_range("2025-07-01", "2026-06-30", freq="D")

rows = []
for centre in CENTRES:
    centre_base = rng.uniform(900, 1500)          # site size varies
    for line in LINES:
        line_factor = {"Plants": 1.6, "Tools": 0.9, "Café": 0.7}[line]
        if line == "Plants":                      # spring peak
            season = 1 + 0.45 * np.sin(2 * np.pi * (dates.dayofyear - 29) / 365)
        else:
            season = np.ones(len(dates))
        weekend = dates.dayofweek >= 5
        weekend_factor = {"Plants": 1.15, "Tools": 1.05, "Café": 1.45}[line]
        sales = (centre_base * line_factor * season
                 * np.where(weekend, weekend_factor, 1.0)
                 * rng.normal(1, 0.10, len(dates)))
        rows.append(pd.DataFrame({"date": dates, "centre": centre,
                                  "line": line, "sales": sales.round(2)}))

daily = pd.concat(rows, ignore_index=True)
daily.to_csv(f"{DATA_DIR}/hartwell_daily.csv", index=False)

# The finance office's monthly report, as it arrives by email: Months as columns
monthly_report = (daily
                  .assign(month=daily["date"].dt.to_period("M").astype(str))
                  .pivot_table(values="sales", index=["centre", "line"],
                               columns="month", aggfunc="sum")
                  .round(0)
                  .reset_index())
monthly_report.to_csv(f"{DATA_DIR}/hartwell_monthly_report.csv", index=False)

print(f"Created {DATA_DIR}/hartwell_daily.csv ({len(daily)} rows)")
print(f"Created {DATA_DIR}/hartwell_monthly_report.csv ({len(monthly_report)} rows)")

The next cell reads the daily file back in, in the way files from a real client would arrive at your desk.

In [ ]:
daily = pd.read_csv(f"{DATA_DIR}/hartwell_daily.csv", parse_dates=["date"])
print(daily.head())
print(f"\n{len(daily)} rows, one per centre, per line, per day")

## 1. Tidy data

The daily file is **tidy**. It meets all three rules of tidy data, which the [Introduction](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/05/Introduction.ipynb) sets out. Its columns are `date`, `centre`, `line`, and `sales`. Each row is one centre, one line, and one day.

Now compare that to the finance office's monthly report:

In [ ]:
report = pd.read_csv(f"{DATA_DIR}/hartwell_monthly_report.csv")
report.head()

It was written for *people*, not *machines*. While people can read it, it is *messy*, at least for a machine. Specifically, it breaks the rule that each variable forms a column. The column headers (`2025-07`, `2025-08`, …) are **not** variable names. They are **values** of a variable called `month`, spread across the header row.

For an [analyst](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-analyst), this shape is problematic:

- "Total sales per centre" means adding up a dozen columns by hand
- Next month a new column appears, and code that ran in July breaks in August
- No plotting library accepts it in this shape

## 2. `melt()`: Wide to long

[`melt()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.melt.html) turns the columns that are really values into rows. It takes these parameters:

- `id_vars`: The columns to keep as identifiers (`centre`, `line`)
- `value_vars`: The columns to turn into rows (by default, every column except `id_vars`)
- `var_name`: The new column for the old headers (`month`)
- `value_name`: The new column for the old cell values (`sales`)

The cell below melts the report, then answers "total per centre" with a single `groupby`:

In [ ]:
tidy_report = report.melt(id_vars=["centre", "line"],
                          var_name="month", value_name="sales")

# Or (remember?):
# tidy_report = report.melt(
#   id_vars=["centre", "line"],
#   var_name="month",
#   value_name="sales"
#   )

print(tidy_report.head())
print(f"\nShape: {report.shape} (wide)  ->  {tidy_report.shape} (long)")

# And now 'total per centre' is one groupby, unaffected by new months:
print("\nTotal sales per centre (£):")
print(tidy_report.groupby("centre")["sales"].sum())

> 🤖 **AI Lens:** Describe the shape you want instead of naming a function. Give your Associate the first prompt below, which describes only the shape. Check that it picks `melt()` without being told. Then give it the second prompt, the reverse (long back to wide). Check that it chooses a pivot.

```text
I have a DataFrame with columns centre, line, and one column per month. I want one row per centre per line per month, with columns centre, line, month, sales.
```

```text
I have a DataFrame with columns centre, line, month, sales: one row per centre per line per month. I want one row per centre per line, with one column per month holding that month's sales.
```

**A good answer** reaches for `melt()` for the first shape and a pivot for the second without being told either verb. The code it writes runs on `report` and `tidy_report` above.

## 3. `pivot_table()`: Long to wide, with aggregation

[`pivot_table()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.pivot_table.html) builds a summary grid from tidy data, and takes these parameters:

- `index`: What goes down the side (**rows =** centres)
- `columns`: What goes across the top (**columns =** product lines)
- `values`: What fills the grid (**cells =** sales)
- `aggfunc`: How to combine the rows that land in one cell (**mean or sum**)

Put together (as one sentence), they make the *shape sentence* that names the **pivot** you want. For example, the cell below builds "rows = centres, columns = lines, cells = average daily sales":

In [ ]:
avg_day = daily.pivot_table(values="sales", index="centre",
                            columns="line", aggfunc="mean").round(0)
avg_day

One line of code turns a year of daily rows into one small grid. *[Magic!](https://youtu.be/cio8ybqqnIM?t=37)*

`aggfunc` also takes a list of functions, or a dictionary that pairs each column with its own function. The optional `margins=True` then adds row and column totals.

We can now build the annual totals in thousands of pounds:

In [ ]:
# Annual totals with margins: A board-ready grid
totals = daily.pivot_table(values="sales", index="centre", columns="line",
                           aggfunc="sum", margins=True, margins_name="Total")
(totals / 1000).round(1)  # in £k, easier to read

### ⚠️ The default `aggfunc`

If you do not set `aggfunc`, `pivot_table()` uses **`"mean"`** as its default. The mean is a statistical choice, and pandas makes it for you. That's often great, and sometimes it isn't: It is *wrong* whenever the rows that end up in one cell should be **added** rather than averaged.

The daily file has three rows per centre and per date, one per product line. Ask for, say, "daily takings by centre" (something a client may ask for) without `aggfunc`, and each cell contains the *average line* instead of the *sum of the lines*:

In [ ]:
wrong = daily.pivot_table(values="sales", index="date", columns="centre")
right = daily.pivot_table(values="sales", index="date", columns="centre",
                          aggfunc="sum")

print("Default aggfunc='mean' gives the average PER LINE:")
print(wrong.head(3).round(0))
print("\naggfunc='sum' gives the till total the client means by 'daily takings':")
print(right.head(3).round(0))
print(f"\nThe silent default understates takings {(right / wrong).mean().mean():.1f}x")

Both cells run, and both tables look **plausible**. This is important: Only one matches what the client means by "daily takings". As a general rule of thumb, always state `aggfunc` (whether you're after the average or not). Then check one cell by hand: Filter the raw rows for one centre and date, and add them up. The [quest](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Guides/glossary.md#term-quest) sets you the same challenge.

> 🤖 **AI Lens:** Associates use this default too. Give yours the vague first prompt. Read the code it writes. Did it state `aggfunc`? Did it ask whether the cells should be sums or means? Then give it the second prompt, the full shape sentence. Compare the two answers. The aggregation is business logic, and stating it is **your** job.

```text
pivot the daily data so I can see sales by date and centre
```

```text
Pivot the daily data so I can see sales by date and centre: rows = dates, columns = centres, cells = the sum of the three lines' sales.
```

**A good answer** to the vague prompt asks you what the cells should contain. A draft that silently takes the mean is the default at work. The full shape sentence should produce `aggfunc="sum"` without any further hint.

## 4. `MultiIndex`: Hierarchical labels

Group by two keys and the result has a [**`MultiIndex`**](https://pandas.pydata.org/docs/user_guide/advanced.html). Its row labels have levels (`centre`, then `line`). The [Introduction](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/05/Introduction.ipynb) covers the idea. Build one first:

In [ ]:
by_cl = daily.groupby(["centre", "line"])["sales"].sum().round(0)
print(by_cl)
print(f"\nIndex levels: {by_cl.index.names}")

The moves you need, in the order the cell below shows them:

1. Plain `.loc` slices the *outer* level and returns one centre's lines
2. A single cell needs both labels together, as the tuple `(centre, line)`
3. [`.xs()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.xs.html) ("cross-section") slices an *inner* level

In [ ]:
# Move 1: Slice the outer level (one centre's lines)
print("Egham only:")
print(by_cl.loc["Egham"])

# Move 2: A single cell needs the full tuple
print(f"\nEgham Café total: £{by_cl.loc[('Egham', 'Café')]:,.0f}")

# Move 3: Slice an INNER level with .xs()
print("\nCafé across all centres (xs on the inner level):")
print(by_cl.xs("Café", level="line"))

### `unstack()` and `stack()`

`unstack()` rotates the innermost index level out into columns, turning the labelled list into a grid. `stack()` reverses it. They reshape data that already has a `MultiIndex`. `melt()` and `pivot_table()` work from plain columns instead. The pandas [reshaping guide](https://pandas.pydata.org/docs/user_guide/reshaping.html) covers all of these functions. The next cell unstacks the totals and stacks them back:

In [ ]:
grid = by_cl.unstack()          # inner level ('line') becomes columns
print(grid)

back = grid.stack()             # and back again
print(f"\nstack() restores the MultiIndex Series: {back.index.names}")

### Missing combinations in `unstack()`

When a combination has **no data**, `unstack()` fills that cell with `NaN`, which means "nothing observed". Suppose the Staines café closes for building work, so the file has no Café rows for Staines:

In [ ]:
subset = daily[~((daily["centre"] == "Staines") & (daily["line"] == "Café"))]
holey = subset.groupby(["centre", "line"])["sales"].sum().round(0).unstack()
holey

That `NaN` is **not zero**. £0 means "the café traded and took no money". `NaN` means "nothing to report". `mean()` skips `NaN`. Replace the `NaN` with `0` and `mean()` counts that zero. Use `unstack(fill_value=0)` only when zero is the true value of an unobserved cell. Zero is usually the true value for a count of events, and rarely for takings. The Debrief returns to this distinction.

> 🤖 **AI Lens:** After any reshape, ask your Associate the question below. Paste the reshape code and its output in place of the placeholder. If it answers 'missing data' without saying *what was not observed*, ask it again.

```text
<paste the reshape code and its output here>

What is the index of this object right now, and what does a NaN in it mean in business terms?
```

**A good answer** names the index and the columns by level. It also says what the `NaN` stands for in business terms, rather than 'missing data'. In this data, it means a centre-and-line combination that was never observed, not a zero take.

## 5. Window functions: Trend from noise

Hartwell's owner is convinced "Plants are dying off". Daily data is too noisy to answer that by eye. [Window functions](https://pandas.pydata.org/docs/user_guide/window.html) compute each day's value from the days up to and including that day:

- `rolling(7).mean()`: The 7-day moving average (the smoothed line)
- `shift(7)`: The value 7 rows back (this day last week)
- `pct_change(7)`: The percentage change against 7 rows back

They all work on row order, so **sort by date first**. A rolling average over unsorted rows is meaningless.

The next cell totals Plants across the three centres per day, then adds a 28-day rolling mean:

In [ ]:
plants = (daily[daily["line"] == "Plants"]
          .groupby("date")["sales"].sum()        # all three centres together
          .sort_index())

trend = pd.DataFrame({
    "daily": plants,
    "rolling_28": plants.rolling(28).mean(),
})
print(trend.round(0).iloc[26:32])   # the first 27 rolling values are NaN. Why?

The numbers point to a seasonal peak, but the owner will believe a picture. Plot the daily series in grey with the 28-day mean on top, using [`DataFrame.plot()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.plot.html):

In [ ]:
ax = trend.plot(figsize=(10, 4), color=["lightgrey", "crimson"],
                title="Plants: daily sales vs 28-day rolling mean")
ax.set_ylabel("sales (£)")
plt.show()

The grey `daily` spikes are weekend peaks and random day-to-day variation. The red `rolling_28` line answers the owner's question. Plants have a spring peak, not a die-off. These details matter here:

- The first 27 rolling values are `NaN`, because a 28-day window needs 28 days of history. `min_periods` relaxes that if you need it
- `rolling()` smooths only the series you give it. Smoothing per group needs `groupby()` plus [`transform()`](https://pandas.pydata.org/docs/reference/api/pandas.api.typing.DataFrameGroupBy.transform.html), which the next cell uses for week-on-week change per centre

Here `lambda s: s.shift(7)` is a one-line unnamed function that shifts each centre's sales by seven rows (optional background: [Week 2b](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/02b/Introduction.ipynb)).

In [ ]:
# shift() and pct_change(): Week-on-week change, per centre, via transform()
centre_daily = (daily.groupby(["date", "centre"], as_index=False)["sales"].sum()
                .sort_values(["centre", "date"]))

# transform() applies the window WITHIN each centre and returns aligned rows
centre_daily["last_week"] = centre_daily.groupby("centre")["sales"].transform(
    lambda s: s.shift(7))
centre_daily["wow_%"] = centre_daily.groupby("centre")["sales"].transform(
    lambda s: (s.pct_change(7) * 100).round(1))

latest = centre_daily[centre_daily["date"] == "2026-06-30"]
print(latest.round({"sales": 0, "last_week": 0}))

A plain `shift(7)` on the whole table would put Staines' last seven days onto the first Windsor rows. The window crosses from one centre into the next. `transform()` runs the window inside each group and returns a column aligned to the original rows.

> 🤖 **AI Lens:** Ask your Associate the first question below, with the code of the `transform()` cell pasted in place of the placeholder. Then give it the second prompt, which asks for a tiny counter-example that proves the answer. Make the Associate *demonstrate* a claim, not only assert it.

```text
<paste the cell's code here>

Explain the difference between `df['sales'].shift(7)` and `df.groupby('centre')['sales'].shift(7)` on this data. What exactly goes wrong with the first?
```

```text
Construct a tiny DataFrame with two centres and at least eight days each, sorted by centre then date, and show `shift(7)` with and without `groupby('centre')` side by side. Point at the rows where the plain shift hands one centre's values to the other.
```

**A good answer** says the plain shift hands the last rows of one centre to the first rows of the next. It also says the grouped shift stays inside each centre. The counter-example shows the leaked rows rather than asserting them.

## 6. Complex `groupby()`

Client <!-- plain-language: everyday brief -->briefs often ask for several statistics at once. `agg()` accepts a dictionary that maps each column to a function, or to a list of functions. [**Named aggregation**](https://pandas.pydata.org/docs/user_guide/groupby.html#named-aggregation) is tidier. Each output column gets a readable name. The next cell shows both forms:

In [ ]:
# Dictionary form: Quick, but the output has stacked column labels
dict_form = daily.groupby("line").agg({"sales": ["mean", "sum", "max"]}).round(0)
print(dict_form)

# Named aggregation: Flat, readable, board-friendly column names
named_form = daily.groupby("line").agg(
    typical_day=("sales", "mean"),
    annual_total=("sales", "sum"),
    best_day=("sales", "max"),
).round(0)
print("\n", named_form)

`transform()` also copies a group statistic back onto every row. Here it gives each line's share of its centre's takings, which answers "which line earns most for each centre?":

In [ ]:
# transform() again, this time for shares: Which line earns most?
mix = daily.groupby(["centre", "line"], as_index=False)["sales"].sum()
mix["share_%"] = (mix["sales"]
                  / mix.groupby("centre")["sales"].transform("sum")
                  * 100).round(1)

# ...and unstack the shares into a readable grid
mix.set_index(["centre", "line"])["share_%"].unstack()

## 7. Putting it together: Method chaining

Professional pandas reads as a pipeline. Each line is one verb, top to bottom, with no `df2`, `df3`, or `df_final_v2`. The next cell repeats this notebook's steps in one chain, from raw daily rows to a quarterly board grid. [`assign()`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.assign.html) adds the quarter column:

In [ ]:
board_grid = (
    daily
    .assign(quarter=daily["date"].dt.to_period("Q").astype(str))
    .groupby(["quarter", "line"])["sales"].sum()
    .div(1000)                        # £k
    .round(1)
    .unstack("line")                  # lines across the top
)
board_grid

Read it aloud: *Take the daily data, tag each row with its quarter, total sales by quarter and line, in thousands, lines across the top.*

## Next steps

You have every tool the quest needs:

- `pivot_table` with an explicit `aggfunc`
- A comparison table from `pivot_table`, with one field down the side and one across the top
- Windows for trend
- The habit of verifying one cell by hand

Open [Quest.ipynb](https://github.com/REPPL/Teaching/blob/main/MN5813/2026-27/Week/05/Quest.ipynb).